In [2]:
import os 

os.chdir('../..')
os.getcwd()

'/home/leostre/Рабочий стол/SoftStairs-QAT'

In [3]:
from ultralytics import YOLO 

MODEL = 'yolo26n.pt'
DATASET = "HomeObjects-3K.yaml"

N_EPOCHS = 100
IMGSIZE = 640

In [8]:
from torch.ao.quantization import get_default_qat_qconfig, convert, prepare_qat 

In [ ]:


# # 4. Train or fine-tune the model as usual
# train_loop(model)

# # 5. Convert the model to a truly quantized version for inference
# quantize_(model, QATConfig(base_config, step="convert"))

# The model is now ready for efficient inference

baseline no qat

In [31]:
model = YOLO("yolo26n.pt")
results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    save_period=10,
    name='no-qat'
)

Ultralytics 8.4.107 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15832MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=no-qat, nbs=64, nms=False,

In [27]:
model = YOLO("yolo26n.pt")


import torch
from torchao.quantization import quantize_, Int8WeightOnlyConfig
from torchao.quantization.qat import QATConfig

# 1. Load or define your model
model
# 2. Define the quantization configuration
# This example uses int8 activations and int4 weights, a common scheme [citation:1][citation:6]
base_config = Int8WeightOnlyConfig(group_size=32)

# 3. Prepare the model for QAT (adds "fake" quantization)
quantize_(model, QATConfig(base_config, step="prepare"))

results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    name='qat-Int8',
    save_period=10 
)


Ultralytics 8.4.107 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15832MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=qat-Int8, nbs=64, nms=Fals

Traceback (most recent call last):
  File "/usr/lib/python3.10/multiprocessing/queues.py", line 239, in _feed
    reader_close()
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 177, in close
    self._close()
  File "/usr/lib/python3.10/multiprocessing/connection.py", line 361, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor


      2/100      4.02G      1.287      2.722   0.008777        212        640: 100% ━━━━━━━━━━━━ 143/143 16.7it/s 8.6s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 21.5it/s 0.6s0.1s
                   all        404       3470      0.554      0.358       0.34      0.235

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      3/100      4.02G      1.304      2.364   0.008872        141        640: 100% ━━━━━━━━━━━━ 143/143 18.0it/s 8.0s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 13/13 23.3it/s 0.6s0.1s
                   all        404       3470      0.495      0.393      0.385      0.267

      Epoch    GPU_mem   box_loss   cls_loss    l1_loss  Instances       Size
      4/100      4.02G      1.282      2.145   0.008889        136        640: 100% ━━━━━━━━━━━━ 143/143 18.5it/s 7.7s0.1s
                 Class     Imag

In [32]:
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig

def get_qconfig(strategy, t_start, steps):
    return QuantizationConfig(
        n_bits=8, normalized=True, t_scheduler_strategy=strategy, t_start=t_start, t_end=1e-4, t_step=steps
    )

def run_experiment(model_name, n_epochs, strategy, t_start):
    model = YOLO(model_name) 
    qconfig = get_qconfig(strategy, t_start, n_epochs)
    quantizer = SoftStairsQuantizer(model, qconfig, total_steps=n_epochs, excluded_modules={n for n, p in model.named_modules() if 'bn' in n})
    def qstep(*args, **kwargs):
        quantizer.step()
    
    model.add_callback('on_fit_epoch_end', qstep)
    model.train(data=DATASET, epochs=n_epochs, name=f'{strategy}-{t_start}', imgsz=IMGSIZE, 
    save_period=10 )





In [ ]:
from itertools import product 

STRATEGIES = ['linear', 'cos', 'step', 'exp', 'constant', ]
T_START = [0.5, 0.1]

for strat, t in product(STRATEGIES, T_START):
    print('Running:', strat, t)
    run_experiment(MODEL, n_epochs=N_EPOCHS, strategy=strat, t_start=t)

Running: constant 0.5
Ultralytics 8.4.107 🚀 Python-3.10.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA GeForce RTX 5080, 15832MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=HomeObjects-3K.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=cons

KeyboardInterrupt: 

In [ ]:
from softstairs_qat import SoftStairsQuantizer, QuantizationConfig

def get_qconfig(strategy, t_start, steps):
    return QuantizationConfig(
        n_bits=8, normalized=True, t_scheduler_strategy=strategy, t_start=t_start, t_end=1e-4, t_step=steps
    )

def run_experiment_nb(model_name, n_epochs, strategy, t_start):
    model = YOLO(model_name) 
    qconfig = get_qconfig(strategy, t_start, n_epochs)
    qconfig.n_bits = 4
    quantizer = SoftStairsQuantizer(model, qconfig, total_steps=n_epochs, excluded_modules={n for n, p in model.named_modules() if 'bn' in n})
    def qstep(*args, **kwargs):
        quantizer.step()
    
    model.add_callback('on_fit_epoch_end', qstep)
    model.train(data=DATASET, epochs=n_epochs, name=f'{strategy}-{t_start}-{qconfig.n_bits}', imgsz=IMGSIZE, 
    save_period=10 )


In [ ]:
from itertools import product 

STRATEGIES = ['constant', 'linear', 'cos', 'step', 'exp', ]
T_START = [0.5, 0.25, 0.1]

for strat, t in product(STRATEGIES, T_START):
    print('Running:', strat, t)
    run_experiment_nb(MODEL, n_epochs=N_EPOCHS, strategy=strat, t_start=t)

In [ ]:
from torchao.quantization import quantize_, Int4WeightOnlyConfig

# 1. Load or define your model
model = YOLO("yolo26n.pt")
# 2. Define the quantization configuration
# This example uses int8 activations and int4 weights, a common scheme [citation:1][citation:6]
base_config = Int4WeightOnlyConfig(group_size=32)

# 3. Prepare the model for QAT (adds "fake" quantization)
quantize_(model, QATConfig(base_config, step="prepare"))

results = model.train(
    data=DATASET,
    epochs=N_EPOCHS,
    imgsz=IMGSIZE,
    name='qat-Int4',
    save_period=10 
)